In [5]:
import pandas as pd

In [6]:
def get_closest_row(df, target_time):
    df = df.copy()
    df["time_diff"] = (df["tracked_at"] - target_time).abs()
    return df.sort_values("time_diff").iloc[0]

In [7]:
def build_dataset():

    # ===== LOAD & GABUNG COHORT =====
    cohort1 = pd.read_csv("cohort.csv")
    cohort2 = pd.read_csv("cohort2.csv")

    cohort1["cohort_id"] = "cohort1"
    cohort2["cohort_id"] = "cohort2"

    cohort = pd.concat([cohort1, cohort2], ignore_index=True)

    # ===== LOAD & GABUNG TRACKING =====
    tracking1 = pd.read_csv("tracking_log.csv")
    tracking2 = pd.read_csv("tracking_log2.csv")

    tracking1["cohort_id"] = "cohort1"

    tracking = pd.concat([tracking1, tracking2], ignore_index=True)

    # ===== CONVERT DATETIME =====
    cohort["published_at"] = pd.to_datetime(cohort["published_at"], utc=True)
    tracking["tracked_at"] = pd.to_datetime(tracking["tracked_at"], utc=True)

    final_rows = []

    # ===== BUILD DATASET =====
    for vid in cohort["video_id"].unique():

        vid_track = tracking[tracking["video_id"] == vid]

        if vid_track.empty:
            continue

        try:
            row_cohort = cohort[cohort["video_id"] == vid].iloc[0]

            start_time = pd.to_datetime(row_cohort["published_at"])

            target_6h = start_time + pd.Timedelta(hours=6)
            target_24h = start_time + pd.Timedelta(hours=24)

            row_6h = get_closest_row(vid_track, target_6h)
            row_24h = get_closest_row(vid_track, target_24h)

            final_rows.append({

                "video_id": vid,
                "cohort_id": row_cohort["cohort_id"],

                "views_6h": int(row_6h["views"]),
                "likes_6h": int(row_6h["likes"]),
                "comments_6h": int(row_6h["comments"]),

                "views_24h": int(row_24h["views"]),
                "likes_24h": int(row_24h["likes"]),
                "comments_24h": int(row_24h["comments"]),
            })

        except Exception as e:
            print(f"Skip {vid} karena error: {e}")
            continue

        
    # ===== FINAL DATAFRAME =====
    df_final = pd.DataFrame(final_rows)

    df_final = df_final.drop_duplicates(subset=["video_id"])

    # ===== SAVE =====
    df_final.to_csv("youtube_dataset.csv", index=False)

    print(f"✅ Dataset berhasil dibuat: {len(df_final)} video")

In [8]:
if __name__ == "__main__":
    build_dataset()

✅ Dataset berhasil dibuat: 247 video
